In [8]:
import os
import torch
import pandas as pd
import scanpy as sc
from sklearn import metrics
import multiprocessing as mp
import numpy as np
from GraphST import GraphST
from GraphST.utils import clustering
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

#folder_list = ["scenario2_1/p_15000", "scenario2_1/p_20000", "scenario2_1/p_25000", "scenario2_1/p_30000"]
folder_list = ["scenario2_1/p_10000"]

In [9]:
for folder in folder_list:
    if not os.path.exists(folder + "/" + "GraphST"):
        os.makedirs(folder + "/" + "GraphST")



    result = pd.DataFrame(columns = ["ARI", "AMI", "NMI"])

    for data_index in range(30):
        data = str(data_index + 1)
        file_name = folder + "/"+ "SpaGCN/data/" + data +".h5ad"
        adata = sc.read_h5ad(file_name)
        adata.X = adata.X.astype("float32")
        adata.obsm["spatial"] = pd.concat([adata.obs["row"], adata.obs["col"]], axis=1)
        adata.var["highly_variable"] = True


        # model = GraphST.GraphST(adata, device=device)
        # adata = model.train()
        # # set radius to specify the number of neighbors considered during refinement
        # radius = 1.414

        # tool = 'louvain' # mclust, leiden, and louvain
        # n_clusters = np.unique(adata.obs["label"]).size

        # # clustering
        # if tool == 'mclust':
        #     clustering(adata, n_clusters, radius=radius, method=tool, refinement=True) # For DLPFC dataset, we use optional refinement step.
        # elif tool in ['leiden', 'louvain']:
        #     clustering(adata, n_clusters, radius=radius, method=tool, start=0.1, end=2.0, increment=0.01, refinement=False)

        from sklearn import metrics
        # adata.obs["pred"] = adata.obs["domain"]
        obs_df = adata.obs.dropna()
        pred = pd.read_csv(folder + "/GraphST/" + data + ".csv", index_col=0)
        obs_df["pred"] = pred
        ari = metrics.adjusted_rand_score(obs_df['pred'], obs_df['label'])
        nmi = metrics.normalized_mutual_info_score(obs_df['pred'], obs_df['label'])
        ami = metrics.adjusted_mutual_info_score(obs_df['pred'], obs_df['label'])

        values= [ari, ami, nmi]
        result.loc[data_index] = values

        #obs_df["pred"].to_csv(folder + "/GraphST/" + data + ".csv")

    result_file = folder + "/summary/" + "GraphST.csv"
    result.to_csv(result_file )

In [7]:
folder_list

['scenario2_1/p_15000']

In [12]:
folder = "scenario3_2"
result = pd.DataFrame(columns = ["ARI", "AMI", "NMI"])
data_index  = 0
data = str(data_index + 1)
file_name = folder + "/"+ "SpaGCN/data/" + data +".h5ad"
adata = sc.read_h5ad(file_name)
adata.X = adata.X.astype("float32")
adata.obsm["spatial"] = pd.concat([adata.obs["row"], adata.obs["col"]], axis=1)
adata.obs["Ground Truth"] = adata.obs["label"]
adata.var["highly_variable"] = True

In [13]:
adata.var["highly_variable"]

gene-1       True
gene-2       True
gene-3       True
gene-4       True
gene-5       True
             ... 
gene-1996    True
gene-1997    True
gene-1998    True
gene-1999    True
gene-2000    True
Name: highly_variable, Length: 2000, dtype: bool

In [16]:
# define model
model = GraphST.GraphST(adata, device=device)
adata = model.train()

Begin to train ST data...


100%|██████████| 600/600 [00:04<00:00, 139.55it/s]

Optimization finished for ST data!


In [15]:
adata

AnnData object with n_obs × n_vars = 1600 × 2000
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'row', 'col', 'label', 'Ground Truth'
    var: 'features', 'highly_variable'
    obsm: 'X_PCA', 'spatial'